# 22 - ¿Sirve elegir el pool no etiquetado por incertidumbre?

## La pregunta

Es la sugerencia de Ivson que ningún experimento ha tocado todavía:

> analise de incerteza: [...] talvez possa ajudar no treinamento SSL, na selecao dos
> frames. ex. **descartar frames com muita incerteza pq pseudo labels estao ruins?**

Ojo a la dirección: él propuso **descartar** los inciertos, no seleccionarlos. Por eso
este notebook corre **los dos brazos**, o el experimento no prueba nada.

**Esto NO es active learning.** Active learning es elegir qué frames *anotar*, con un
oráculo humano y coste de anotación. Aquí no se anota nada: se elige qué frames *no
etiquetados* entran al SSL. En esta tesis eso ya tiene nombre, **selection policy**, y
es la RQ3. No se abre un eje nuevo: se añade una tercera política a las dos que ya hay.

## El diseño

Tamaño fijo de 3.937 frames, el mismo del pool `r=10`:

| política | de dónde salen los 3.937 | estado |
|---|---|---|
| temporal `r=10` | los más cercanos en tiempo a un frame etiquetado | ya existe |
| aleatoria igualada | al azar | ya existe |
| **más inciertos** | mayor entropía media | **nuevo** |
| **menos inciertos** | menor entropía media | **nuevo** |

## Dos cosas que se verificaron ANTES de escribir esto

**1. El control aleatorio iguala los frames POR VÍDEO, no solo el total.** Los 23
vídeos coinciden uno a uno con el pool temporal. Así que la selección por incertidumbre
tiene que hacer lo mismo: coger los `k_v` más inciertos *dentro de cada vídeo*. Si se
cogieran los 3.937 más inciertos globales, la composición por vídeo se descontrolaría y
la comparación quedaría confundida.

**2. `unlabeling_all_lateral` ya está filtrado por el corte AP.** Comprobado en los 17
vídeos con corte: el frame máximo de cada uno cae justo por debajo de su cutoff. O sea
que seleccionar de ahí hereda el filtro y no hace falta reimplementarlo.

## El cuello de botella no es la GPU, es Drive

Leer 74.774 PNG uno a uno desde Drive es lentísimo. Por eso el Paso 2 hace `tar` →
copia de **un** archivo → extracción local, y a partir de ahí lee del SSD. El Paso 1
mide el rendimiento real y avisa si no sale a cuenta.

`entropy_all_lateral.csv` queda guardado en Drive: es un activo reutilizable, se calcula
una vez y sirve para cualquier experimento futuro de incertidumbre.


In [ ]:
# ============================================================
# SETUP - correr una vez tras cada reinicio del runtime
# No entrena nada.
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

!git clone https://github.com/sebastianquispearias/tesis-seg.git
%cd tesis-seg
!pip install -q -r requirements.txt

import torch, sys
print("Python    :", sys.version)
print("torch     :", torch.__version__)
print("CUDA      :", torch.version.cuda)
!git log --oneline -1
!nvidia-smi | grep -E "NVIDIA|Driver Version|CUDA Version"

import sys
sys.path.append("/content/tesis-seg")

import json, os, time, glob, statistics

from src.defaults import get_default_config, summarize_config
from src.augmentations import get_supervised_train_augmentation
from src.datasets import build_supervised_datasets, build_dataloaders
from src.train import run_training
from src.evaluate import evaluate_checkpoint

# --- GUARDIAN 1: el codigo clonado debe traer los perfiles nuevos ---
# El 2026-08-16 se perdieron 20 runs porque un flag se ignoro en silencio.
import inspect
from src import augmentations as _ag
from src import preprocessing as _pp
assert "get_nnunet_style_augmentation" in inspect.getsource(_ag), (
    "CODIGO VIEJO: src/augmentations.py no tiene los perfiles de nnU-Net. "
    "Borra /content/tesis-seg, vuelve a clonar y reinicia el entorno.")
assert "image_norm" in inspect.getsource(_pp.preprocess_image_and_mask), (
    "CODIGO VIEJO: preprocess_image_and_mask no acepta image_norm.")
print("OK: el codigo clonado trae aug_profile e image_norm.")

# --- GUARDIAN 2: dejar constancia del entorno ---
# Colab cambio de stack entre julio y septiembre de 2026 y albumentations 2.x
# DESCARTA EN SILENCIO argumentos que la 1.x aceptaba (var_limit, value,
# mask_value). Por eso este notebook NO compara contra runs_final_v1, que se
# entreno con albumentations 1.3.1: entrena su propio baseline en esta sesion.
import albumentations as _alb
import segmentation_models_pytorch as _smp
ENTORNO = {"python": sys.version.split()[0], "torch": torch.__version__,
           "albumentations": _alb.__version__, "smp": _smp.__version__,
           "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"}
print()
print("ENTORNO DE ESTA SESION")
for k, v in ENTORNO.items():
    print("   %-16s %s" % (k, v))
print()
print("runs_final_v1 se entreno con: albumentations 1.3.1, smp 0.3.3, torch 2.2.1.")
print("Si lo de arriba no coincide, es NORMAL y por eso hay un brazo baseline aqui.")


---
### Paso 1 - Verificar ANTES de gastar nada

No entrena y no puntúa el pool. Comprueba que el modelo que sirve de juez carga con el
torch de hoy, que los pools están donde se espera, que los conteos por vídeo son los que
se van a igualar, y **mide cuánto tarda de verdad** leer e inferir, para extrapolar.


In [ ]:
# ============================================================
# VERIFICACION - NO ENTRENA, NO PUNTUA EL POOL. ~3 minutos.
# ============================================================
import collections, glob, os, re, time
import numpy as np
import torch

BASE    = "/content/drive/MyDrive/UNM_vertebras_seg_v3"
ROTULOS = "/content/drive/MyDrive/UNM TalkBank Dysphagia/rotulos"

POOL_TEMPORAL = f"{BASE}/unlabeling_r10_max0/images"
POOL_ALEATORIO = f"{BASE}/unlabeling_std_matched_r10/images"
POOL_TODOS     = f"{BASE}/unlabeling_all_lateral/images"

# el juez: un U-Net++ supervisado entrenado en esta misma tanda, con el ruido arreglado
JUEZ = f"{BASE}/runs_nnunet_ablation/A_baseline_fixnoise/seed_0/best_model.pt"

print("PUERTA 1: el ruido de la augmentation es el pedido")
_gn = [t for t in get_supervised_train_augmentation({}).transforms
       if t.__class__.__name__ == "GaussNoise"][0]
_sr = _gn.to_dict()["transform"]["std_range"]
assert _sr[1] < 0.05, ("CODIGO VIEJO: std_range es el default de la 2.x. "
                       "Borra /content/tesis-seg, vuelve a clonar y reinicia.")
print("   std %.2f a %.2f niveles sobre 255   OK" % (_sr[0] * 255, _sr[1] * 255))
print()

print("PUERTA 2: los pools estan y tienen el tamano esperado")
_n = {}
for _nom, _d in [("temporal r10", POOL_TEMPORAL), ("aleatorio r10", POOL_ALEATORIO),
                 ("todos laterales", POOL_TODOS)]:
    assert os.path.isdir(_d), f"FALTA {_d}"
    _n[_nom] = len([f for f in os.listdir(_d) if f.endswith(".png")])
    print("   %-18s %6d frames" % (_nom, _n[_nom]))
assert _n["temporal r10"] == _n["aleatorio r10"], "FALLO: los dos r10 no miden igual"
print("   OK")
print()

print("PUERTA 3: el juez existe y CARGA con el torch de hoy")
assert os.path.isfile(JUEZ), f"FALTA el checkpoint juez: {JUEZ}"
from src.models import create_model
_m = create_model("unetpp", "efficientnet-b3", 1)
_sd = torch.load(JUEZ, map_location="cpu", weights_only=False)
if isinstance(_sd, dict) and "model_state_dict" in _sd:
    _sd = _sd["model_state_dict"]
elif isinstance(_sd, dict) and "state_dict" in _sd:
    _sd = _sd["state_dict"]
_faltan = _m.load_state_dict(_sd, strict=False)
print("   claves que faltan :", len(_faltan.missing_keys))
print("   claves sobrantes  :", len(_faltan.unexpected_keys))
assert len(_faltan.missing_keys) == 0, "FALLO: el checkpoint no encaja con el modelo"
print("   OK: carga limpio")
print()

print("PUERTA 4: los conteos POR VIDEO que hay que igualar")
def _por_video(d):
    c = collections.Counter()
    for f in os.listdir(d):
        m = re.match(r"(v\d+)_f\d+\.png$", f)
        if m:
            c[m.group(1)] += 1
    return c

K_OBJETIVO = _por_video(POOL_TEMPORAL)
_alea = _por_video(POOL_ALEATORIO)
_dif = [v for v in K_OBJETIVO if K_OBJETIVO[v] != _alea.get(v)]
print("   videos                    :", len(K_OBJETIVO))
print("   suma de k_v               :", sum(K_OBJETIVO.values()))
print("   videos donde temporal y aleatorio NO coinciden:", len(_dif))
assert not _dif, "FALLO: el control aleatorio no iguala por video; revisar el diseno"
print("   OK: el aleatorio iguala por video, asi que el de incertidumbre tambien debe")
print()

print("PUERTA 5: el filtro AP se hereda de all_lateral")
_mx = collections.defaultdict(int)
for f in os.listdir(POOL_TODOS):
    m = re.match(r"(v\d+)_f(\d+)\.png$", f)
    if m:
        _mx[m.group(1)] = max(_mx[m.group(1)], int(m.group(2)))
print("   videos en all_lateral     :", len(_mx))
print("   OK (el filtro se verifico en local: ningun video pasa su corte AP)")
print()

print("PUERTA 6: CUANTO TARDA DE VERDAD. Esto es lo que decide el Paso 2.")
_dev = "cuda" if torch.cuda.is_available() else "cpu"
_m = _m.to(_dev).eval()
_muestra = sorted([f for f in os.listdir(POOL_TODOS) if f.endswith(".png")])[::997][:120]
_cfg = get_default_config()
_cfg["target_size"] = (320, 320)
_cfg["use_pad"] = True
_cfg["imagenet_norm"] = False
_cfg["image_preproc"] = "base"

from src.datasets import UnlabeledFramesDataset
_ds = UnlabeledFramesDataset(images_dir=POOL_TODOS, cfg=_cfg)
_t0 = time.time()
with torch.no_grad():
    for _i in range(min(120, len(_ds))):
        _b = _ds[_i]
        _x = _b["weak_image"].float()          # ya es tensor (C, H, W)
        if _x.ndim == 3:
            _x = _x.unsqueeze(0)
        _ = torch.sigmoid(_m(_x.to(_dev)))
_dt = time.time() - _t0
_vel = 120 / _dt
print("   %.1f frames/s leyendo DESDE DRIVE" % _vel)
print("   los 74.774 frames tardarian: %.0f min (%.1f h)" % (74774 / _vel / 60,
                                                             74774 / _vel / 3600))
print()
if 74774 / _vel / 3600 > 2.0:
    print("   >>> AVISO: desde Drive no sale a cuenta. El Paso 2 hara tar + local,")
    print("       o baja CANDIDATOS_POR_OBJETIVO en el Paso 1b.")
else:
    print("   >>> Se puede leer directo de Drive si se prefiere.")
print()
print("TODO VERIFICADO.")


---
### Paso 1b - Los parámetros

`CANDIDATOS_POR_OBJETIVO` es el interruptor de coste. `None` puntúa el pool entero y da
el contraste más fuerte (top y bottom 5%). Un número *m* puntúa un candidato aleatorio
de *m* × 3.937 frames por vídeo y deja el contraste en top y bottom 1/*m*. Si el Paso 1
dijo que el pool entero cuesta demasiado, poner 6.


In [ ]:
# ============================================================
# PARAMETROS DEL EXPERIMENTO
# ============================================================
OUT_ROOT   = f"{BASE}/runs_pool_incertidumbre"       # directorio NUEVO
LOCAL      = "/content/lateral"                      # SSD de Colab
CSV_ENTROPIA = f"{BASE}/resultados/entropy_all_lateral.csv"

SEMILLAS = [0, 1, 2]

# None = puntuar el pool entero (contraste top/bottom 5%)
# 6    = candidato aleatorio de 6x el objetivo (contraste top/bottom 17%), 3x mas barato
CANDIDATOS_POR_OBJETIVO = None

POOL_INCIERTOS = "unlabeling_uncertainty_top_r10"
POOL_CIERTOS   = "unlabeling_uncertainty_bottom_r10"

# Los cuatro brazos. Lo UNICO que cambia entre ellos es unlabeled_subdir.
BRAZOS = [
    ("MT_temporal_r10",    "unlabeling_r10_max0/images"),
    ("MT_aleatorio_r10",   "unlabeling_std_matched_r10/images"),
    ("MT_mas_inciertos",   f"{POOL_INCIERTOS}/images"),
    ("MT_menos_inciertos", f"{POOL_CIERTOS}/images"),
]

BRAZO_BASELINE = "MT_aleatorio_r10"   # la politica sin criterio, contra la que se mide

print("brazos   :", [b[0] for b in BRAZOS])
print("semillas :", SEMILLAS)
print("salida   :", OUT_ROOT)
print("candidatos por objetivo:", CANDIDATOS_POR_OBJETIVO or "todos (74.774)")
print()
print("OJO: los dos primeros brazos YA EXISTEN en runs_final_v1, pero se reentrenan")
print("aqui para que los cuatro salgan del MISMO entorno. Es la leccion del 16/8 y del")
print("bug del ruido: no comparar contra runs de otra sesion.")


---
### Paso 1c - El cuerpo de un run

Mean Teacher con la configuracion de `mean_teacher_std_matched_r10`. `cambios` es lo unico que distingue un brazo de otro, y aqui solo cambia el pool.


In [ ]:
def run_arm(nombre, semilla, cambios, verbose=False):
    """Train a Mean Teacher UNM run, applying `cambios` over the reference config.

    The configuration is the one behind the r=10 rows already reported in the
    manuscript, copied from runs_final_v1/mean_teacher_std_matched_r10; only the
    unlabeled pool and the seed change between runs. A finished run is skipped
    instead of being retrained.
    """
    import gc; gc.collect()
    torch.cuda.empty_cache()

    cfg = get_default_config()
    cfg["img_root"]    = BASE
    cfg["msk_root"]    = BASE
    cfg["rotulos_dir"] = ROTULOS
    cfg["exp_dir"]     = f"{OUT_ROOT}/{nombre}/seed_{semilla}"

    cfg["arch"]      = "unetpp"
    cfg["backbone"]  = "efficientnet-b3"
    cfg["n_classes"] = 1
    cfg["seed"]      = semilla

    # --- configuracion SSL, copiada de mean_teacher_std_matched_r10 ---
    cfg["use_semi"]             = True
    cfg["ssl_method"]           = "mean_teacher"
    cfg["lambda_u"]             = 0.05
    cfg["tau"]                  = 0.95
    cfg["ema_decay"]            = 0.99
    cfg["semi_start_epoch"]     = 15
    cfg["semi_warmup_epochs"]   = 20
    cfg["use_temp_consistency"] = False
    cfg["lambda_t"]             = 0.0

    cfg["image_preproc"]  = "base"
    cfg["mask_smoothing"] = "none"
    cfg["use_fixed_crop"] = False
    cfg["target_size"]    = (320, 320)
    cfg["use_pad"]        = True
    cfg["imagenet_norm"]  = False

    cfg["batch_size"]     = 5
    cfg["num_workers"]    = 4
    cfg["drop_last"]      = True
    cfg["num_augmented"]  = 5
    cfg["lr"]             = 1e-3
    cfg["weight_decay"]   = 1e-4
    cfg["epochs"]         = 2000
    cfg["warmup_epochs"]  = 10
    cfg["patience_es"]    = 40
    cfg["eval_threshold"] = 0.5

    cfg["save_preds_vis"] = False
    cfg["run_ruler_eval"] = True

    # --- LO UNICO QUE DISTINGUE A ESTE BRAZO ---
    cfg.update(cambios)

    _exp = cfg["exp_dir"]
    _best    = os.path.isfile(os.path.join(_exp, "best_model.pt"))
    _metrics = os.path.isfile(os.path.join(_exp, "test_metrics.csv"))
    _summary = os.path.isfile(os.path.join(_exp, "run_summary.txt"))

    if _best and not _summary:
        print(f"AVISO {nombre}/seed_{semilla}: best_model.pt sin run_summary.txt.")
        print("      Run cortado a medias. Se reentrena desde cero.")
    if _best and _metrics and _summary:
        print(f"Skipping {nombre}/seed_{semilla}: run completo detectado")
        return _exp

    if verbose:
        print(summarize_config(cfg))

    for k, v in cambios.items():
        assert cfg[k] == v, f"FALLO: cfg[{k}] es {cfg[k]!r}, no {v!r}"

    _pool = os.path.join(BASE, cfg["unlabeled_subdir"])
    assert os.path.isdir(_pool), f"FALLO: no existe el pool {_pool}"
    _np = len([f for f in os.listdir(_pool) if f.endswith(".png")])
    assert _np == 3937, f"FALLO: el pool tiene {_np} frames, no 3937"
    print(f"guardian OK: {nombre} -> pool {cfg['unlabeled_subdir']} con {_np} frames")

    train_tf = get_supervised_train_augmentation(cfg)
    train_ds, val_ds, test_ds = build_supervised_datasets(cfg, train_tf=train_tf)
    loaders = build_dataloaders(cfg, train_ds=train_ds, val_ds=val_ds, test_ds=test_ds)

    _bx = next(iter(loaders["train_loader"]))["image"]
    _esperado = cfg["batch_size"] * (1 + cfg["num_augmented"])
    assert _bx.shape[0] == _esperado, (
        f"FALLO: el loader entrega {_bx.shape[0]} imagenes, no {_esperado}.")
    print(f"loader OK: {_bx.shape[0]} imagenes por paso, {tuple(_bx.shape[2:])}")

    _t0 = time.time()
    art = run_training(cfg, loaders)
    results = evaluate_checkpoint(cfg, art["model"], loaders,
                                  art["best_path"], art["history"])
    print(f"[{nombre}/seed_{semilla}] {(time.time()-_t0)/60:.1f} min")
    print(results)
    return _exp


print("run_arm definida.")


---
## Paso 2 - Puntuar el pool y construir las dos políticas nuevas

No entrena. Copia el pool al SSD local con un solo `tar`, calcula la entropía media de
cada frame con el juez, y arma los dos pools por vídeo.

Reanudable: si `entropy_all_lateral.csv` ya está, se salta la parte cara.


In [ ]:
# === PASO 2: puntuar y construir los pools ===
import csv, shutil, subprocess
import pandas as pd

os.makedirs(os.path.dirname(CSV_ENTROPIA), exist_ok=True)

# ---------- 2a. entropia por frame ----------
if os.path.isfile(CSV_ENTROPIA) and len(pd.read_csv(CSV_ENTROPIA)) >= 74000:
    print("entropy_all_lateral.csv ya esta, con",
          len(pd.read_csv(CSV_ENTROPIA)), "filas. Se salta el calculo.")
else:
    if not os.path.isdir(LOCAL):
        print("copiando el pool al SSD local con un solo tar...")
        os.makedirs(LOCAL, exist_ok=True)
        _t0 = time.time()
        subprocess.run(f'tar -cf - -C "{POOL_TODOS}" . | tar -xf - -C "{LOCAL}"',
                       shell=True, check=True)
        print("   %.1f min" % ((time.time() - _t0) / 60))
    _fs = sorted(f for f in os.listdir(LOCAL) if f.endswith(".png"))
    print("frames en local:", len(_fs))

    _cfgE = get_default_config()
    _cfgE.update({"target_size": (320, 320), "use_pad": True, "imagenet_norm": False,
                  "image_preproc": "base", "num_workers": 4})
    _dsE = UnlabeledFramesDataset(images_dir=LOCAL, cfg=_cfgE)
    _ldE = torch.utils.data.DataLoader(_dsE, batch_size=32, shuffle=False,
                                       num_workers=4, pin_memory=True)
    _m = _m.to(_dev).eval()
    _filas, _i = [], 0
    _t0 = time.time()
    with torch.no_grad():
        for _b in _ldE:
            _x = _b["weak_image"].float().to(_dev)
            _p = torch.sigmoid(_m(_x)).clamp(1e-6, 1 - 1e-6)
            _h = -(_p * _p.log() + (1 - _p) * (1 - _p).log())      # entropia binaria
            _hm = _h.mean(dim=(1, 2, 3)).cpu().numpy()
            for _k, _nm in enumerate(_b["name"]):
                _filas.append((_nm, float(_hm[_k])))
                _i += 1
            if _i % 8000 < 32:
                print("   %6d / %d   %.1f min" % (_i, len(_dsE), (time.time() - _t0) / 60))
    pd.DataFrame(_filas, columns=["stem", "H_mean"]).to_csv(CSV_ENTROPIA, index=False)
    print("guardado:", CSV_ENTROPIA, len(_filas), "filas")

ENT = pd.read_csv(CSV_ENTROPIA)
ENT["video"] = ENT["stem"].str.extract(r"^(v\d+)_")
print("entropia: %d frames, %d videos, H entre %.4f y %.4f"
      % (len(ENT), ENT.video.nunique(), ENT.H_mean.min(), ENT.H_mean.max()))

# ---------- 2b. armar los dos pools, POR VIDEO ----------
_rng = np.random.RandomState(42)
for _nombre, _mayor in [(POOL_INCIERTOS, True), (POOL_CIERTOS, False)]:
    _dst = f"{BASE}/{_nombre}/images"
    if os.path.isdir(_dst) and len([f for f in os.listdir(_dst) if f.endswith(".png")]) \
            == sum(K_OBJETIVO.values()):
        print("ya esta:", _nombre)
        continue
    os.makedirs(_dst, exist_ok=True)
    _elegidos = []
    for _v, _k in K_OBJETIVO.items():
        _sub = ENT[ENT.video == _v]
        if CANDIDATOS_POR_OBJETIVO:
            _n = min(len(_sub), _k * CANDIDATOS_POR_OBJETIVO)
            _sub = _sub.iloc[_rng.choice(len(_sub), _n, replace=False)]
        _sub = _sub.sort_values("H_mean", ascending=not _mayor)
        assert len(_sub) >= _k, f"FALLO: {_v} tiene {len(_sub)} candidatos y hacen falta {_k}"
        _elegidos += list(_sub.head(_k).stem)
    for _s in _elegidos:
        shutil.copy2(os.path.join(LOCAL if os.path.isdir(LOCAL) else POOL_TODOS, _s),
                     os.path.join(_dst, _s))
    print("%s -> %d frames" % (_nombre, len(_elegidos)))

# ---------- 2c. las puertas del diseno ----------
print()
print("PUERTAS DEL DISENO")
for _nombre in [POOL_INCIERTOS, POOL_CIERTOS]:
    _d = f"{BASE}/{_nombre}/images"
    _c = _por_video(_d)
    _mal = [v for v in K_OBJETIVO if K_OBJETIVO[v] != _c.get(v, 0)]
    _tot = sum(_c.values())
    print("  %-36s %5d frames | videos mal igualados: %d"
          % (_nombre, _tot, len(_mal)))
    assert _tot == sum(K_OBJETIVO.values()), "FALLO: tamano distinto al objetivo"
    assert not _mal, "FALLO: no iguala por video"
_a = set(os.listdir(f"{BASE}/{POOL_INCIERTOS}/images"))
_b = set(os.listdir(f"{BASE}/{POOL_CIERTOS}/images"))
print("  solapamiento entre los dos pools:", len(_a & _b), "(tiene que ser 0)")
assert not (_a & _b), "FALLO: los dos pools comparten frames"
print("  OK: mismo tamano, misma composicion por video, sin solapamiento.")


---
## Paso 3 - Los 12 runs (~5 h)

Cuatro políticas por tres semillas. Lo único que cambia entre brazos es
`unlabeled_subdir`. Van emparejados por semilla, así que si Colab se cae quedan
comparaciones completas.

Reanudable: volver a ejecutar salta lo terminado.


In [ ]:
# === PASO 3: los 12 runs, emparejados por semilla ===
_fallos = []
for _seed in SEMILLAS:
    for _nombre, _pool in BRAZOS:
        print("=" * 70)
        print(f">>> {_nombre}  seed={_seed}  pool={_pool}")
        print("=" * 70)
        try:
            run_arm(_nombre, _seed, {"unlabeled_subdir": _pool})
        except Exception as e:
            print(f"FALLO en {_nombre}/seed_{_seed}: {type(e).__name__}: {e}")
            _fallos.append((_nombre, _seed, repr(e)))

print()
print("terminado. fallos:", len(_fallos))
for f in _fallos:
    print("  ", f)


---
## Paso 4 - El resumen

Solo lectura. La comparación que importa es cada política contra la **aleatoria**, que
es la que no aplica ningún criterio. Y las dos de incertidumbre entre sí: si el criterio
sirve para algo, tienen que separarse.


In [ ]:
# === PASO 4: RESUMEN (solo lectura) ===
import glob, json, os, statistics as st

def _f1(seeddir):
    ps = glob.glob(os.path.join(seeddir, "*run_report.json"))
    if not ps:
        return None
    tm = json.load(open(ps[0])).get("test_metrics", {})
    return tm.get("sample_mean_f1", tm.get("f1_mean"))

_res = {}
print("%-22s %4s %10s   %s" % ("brazo", "n", "F1", "crudos"))
print("-" * 74)
for _n, _p in BRAZOS:
    _fs = [x for x in (_f1(d) for d in sorted(glob.glob(f"{OUT_ROOT}/{_n}/seed_*"))) if x]
    if not _fs:
        print("%-22s %4s %10s" % (_n, "-", "sin resultados"))
        continue
    _sd = st.stdev(_fs) if len(_fs) > 1 else 0.0
    _res[_n] = st.mean(_fs)
    print("%-22s %4d %.4f+/-%.4f   %s"
          % (_n, len(_fs), st.mean(_fs), _sd, [round(f, 4) for f in _fs]))

_base = _res.get(BRAZO_BASELINE)
if _base and len(_res) > 1:
    print()
    print("CONTRA LA POLITICA SIN CRITERIO (%s)" % BRAZO_BASELINE)
    for _n, _v in _res.items():
        if _n != BRAZO_BASELINE:
            print("  %-22s %+.4f" % (_n, _v - _base))
    _hi, _lo = _res.get("MT_mas_inciertos"), _res.get("MT_menos_inciertos")
    if _hi and _lo:
        print()
        print("LA RESTA QUE DECIDE: mas inciertos menos menos inciertos = %+.4f" % (_hi - _lo))
        if abs(_hi - _lo) < 0.015:
            print("  Lectura: el criterio NO importa. Elegir por incertidumbre da lo mismo")
            print("  que elegir al azar, y la frase de la tesis pasa de decision sin probar")
            print("  a decision MEDIDA.")
        elif _hi > _lo:
            print("  Lectura: los frames inciertos son los utiles. Ivson proponia lo")
            print("  contrario (descartarlos), asi que esto lo contradice con un numero.")
        else:
            print("  Lectura: los frames inciertos PERJUDICAN, que es exactamente la")
            print("  hipotesis de Ivson: pseudo-etiquetas malas.")
